# TRIBE v2 → short-video engagement (Kaggle)

Runs [neuro-engagement-forecasting](https://github.com/RehanKrish21/Neuro-engagement-forecasts) on a free Kaggle GPU. Meta's TRIBE v2 predicts brain responses to each video, and we test whether they forecast real engagement on [SnapUGC](https://github.com/dasongli1/SnapUGC_Engagement) better than simple baselines.

### One-time setup (right-hand panel → *Settings*)
1. **Accelerator → GPU T4 x2** (or P100). Needs a phone-verified Kaggle account.
2. **Internet → On.**
3. **Add-ons → Secrets → Add secret**: label `HF_TOKEN`, value = your Hugging Face **Read** token (with access to [Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B)). Tick the box to attach it to this notebook.

### Two ways to run
- **Interactive:** run the cells top to bottom. Good for steps 1–3 and a small pilot.
- **Background (up to 12 h, browser can be closed):** set `N` in step 2, then **Save Version → Save & Run All (Commit)**. Results appear in the version's **Output** tab. Use this for the real run.

Kaggle gives ~30 GPU hours/week; the notebook stops by itself before the 12-hour limit and keeps everything finished so far.

*TRIBE v2 is CC BY-NC 4.0 (non-commercial). Check SnapUGC's terms too.*

## 1. Setup
Installs TRIBE v2, this package, `accelerate` and `uv`/`ffmpeg` (TRIBE's speech transcription needs them). Run it first, before anything else imports torch, and no restart is needed.

In [ ]:
import importlib.metadata as md, os, shutil, subprocess, sys, time
NOTEBOOK_START = time.time()

REPO = "https://github.com/RehanKrish21/Neuro-engagement-forecasts.git"
PKGS = [
    "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
    f"neuroengage @ git+{REPO}",
    "pyarrow", "gdown", "accelerate",
    # Preinstalled torchaudio is built for a newer torch; TRIBE pins torch 2.6, and a
    # mismatched torchaudio crashes transformers on import ("undefined symbol").
    "torchaudio==2.6.0",
]

def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

def run(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    raise SystemExit("No GPU: Settings → Accelerator → GPU T4 x2")

if sys.version_info < (3, 11):
    raise SystemExit(f"TRIBE v2 needs Python 3.11+, this image has {sys.version.split()[0]}. "
                     "Settings → Environment → 'Always use latest environment'.")

preloaded = [m for m in ("torch", "numpy") if m in sys.modules]

if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not shutil.which("ffmpeg"):
    run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq ffmpeg"])

ready = (version("tribev2") and version("neuroengage") and version("accelerate")
         and (version("torch") or "").startswith("2.6")
         and (version("torchaudio") or "2.6").startswith("2.6"))
if not ready:
    code = run(["uv", "pip", "install", "--python", sys.executable, *PKGS])
    if code != 0:
        print("\nuv failed, retrying with pip ...\n")
        code = run([sys.executable, "-m", "pip", "install", *PKGS])
    if code != 0:
        raise RuntimeError("Install failed: copy the error lines above and send them over.")
    if preloaded:
        raise SystemExit(f"{preloaded} were imported before installing. Run → Restart & clear cell "
                         "outputs, then run this cell first.")

import torch, numpy, tribev2, neuroengage
from neuroengage.tribe_runner import TribeExtractor  # noqa: F401
import transformers.models.llama.modeling_llama  # noqa: F401  (fails fast on torch/torchaudio mismatches)
print(f"✅ Ready: torch {torch.__version__}, numpy {numpy.__version__}, CUDA {torch.cuda.is_available()}, "
      f"neuroengage {version('neuroengage')}, uv {bool(shutil.which('uvx'))}, ffmpeg {bool(shutil.which('ffmpeg'))}")

## 2. Settings and Hugging Face login
Checks your Llama 3.2 access up front, so a permissions problem shows up here and not 20 minutes into a run.

In [ ]:
from pathlib import Path
import logging

SPLIT = "train"        # SnapUGC split: "train" or "val"
N = 300                # videos to run (start small, check timing, then raise)
SEED = 0
TIME_BUDGET_H = 11.0   # stop extracting before Kaggle's 12 h limit (finished videos are kept)

OUT = Path("/kaggle/working/runs")     # saved with the notebook version (20 GB limit)
DATA = Path("/tmp/snapugc")            # temporary disk, not saved with the notebook
CACHE = Path("/tmp/tribe_cache")
for p in (OUT, DATA, CACHE):
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
for noisy in ("httpx", "urllib3", "huggingface_hub", "filelock"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from huggingface_hub import login, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as exc:
    raise RuntimeError("Add your Hugging Face token: Add-ons → Secrets → HF_TOKEN, "
                       "and tick it for this notebook.") from exc

try:
    hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
    print("✅ Hugging Face login OK, Llama 3.2 access OK")
except Exception as exc:
    raise RuntimeError(
        "Can't download Llama 3.2. Either access isn't approved yet "
        "(huggingface.co/meta-llama/Llama-3.2-3B) or the token is fine-grained without "
        "'read access to public gated repos'. Use a Read token."
    ) from exc

## 3. Smoke test: one video
Loads TRIBE v2 once (the first time downloads several GB of models, ~10–20 min) and runs a 52-second trailer. The loaded `extractor` is reused in step 5.

In [ ]:
import urllib.request
import pandas as pd

sample = Path("/tmp/sample.mp4")
if not sample.exists():
    urllib.request.urlretrieve("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", sample)

t0 = time.time()
extractor = TribeExtractor(cache_folder=CACHE)
print(f"Models loaded in {time.time() - t0:.0f} s (low-memory mode: {extractor.low_memory})")

t0 = time.time()
smoke = extractor.run(pd.DataFrame({"video_id": ["sintel"], "path": [str(sample)]}),
                      OUT / "smoke", overwrite=True)
print(f"✅ TRIBE v2 works: {len(smoke)} predicted seconds in {time.time() - t0:.0f} s "
      "(first run includes one-time model downloads; real timing is measured in step 5)")
smoke.head()

## 4. Download SnapUGC
Pulls the split from the Google Drive links in the [SnapUGC repo](https://github.com/dasongli1/SnapUGC_Engagement/blob/main/dataset/readme.md), unpacks it, and finds the label file and video folder.

If `gdown` fails with a quota error: download the file in your browser, create a Kaggle Dataset from it (*Datasets → New Dataset*), add it to this notebook (*Add-ons → Add Input*), and set `LOCAL_ARCHIVE` to its path under `/kaggle/input/...`.

In [ ]:
import gdown, glob
from collections import Counter

LOCAL_ARCHIVE = None   # e.g. "/kaggle/input/snapugc/train.zip" if you uploaded it as a Kaggle Dataset
FILE_IDS = {"train": "1Mv5Esq5gGuxRTayabRUb5NmHwEN3JdbD", "val": "1iY7qUfqivYzUqUID501dKQDpI3NQil7y"}
dl_dir, unpacked = DATA / "download", DATA / SPLIT
dl_dir.mkdir(exist_ok=True)

if not any(unpacked.rglob("*.mp4")):
    if LOCAL_ARCHIVE:
        archive = Path(LOCAL_ARCHIVE)
    else:
        found = [p for p in dl_dir.iterdir() if p.is_file()]
        archive = found[0] if found else Path(gdown.download(id=FILE_IDS[SPLIT], output=f"{dl_dir}/", quiet=False))
    print(f"Unpacking {archive.name} ({archive.stat().st_size / 1e9:.1f} GB) ...")
    shutil.unpack_archive(str(archive), str(unpacked))
    print(shutil.disk_usage("/tmp"))

csvs = sorted(glob.glob(f"{unpacked}/**/*.csv", recursive=True))
video_dirs = Counter(os.path.dirname(p) for p in glob.glob(f"{unpacked}/**/*.mp4", recursive=True))
print("Label files:", csvs)
print("Video folders:", video_dirs.most_common(3))
LABELS = csvs[0]                          # change if a different CSV holds the labels
VIDEOS = video_dirs.most_common(1)[0][0]
pd.read_csv(LABELS).head()

## 5. Run TRIBE v2 on a stratified subset
Picks `N` videos spread across the engagement range. It times the first 5 to estimate the total, then processes videos in batches until done or until `TIME_BUDGET_H` is reached. Finished videos are never redone.

In [ ]:
from neuroengage.snapugc import build_manifest, load_labels

RUN = OUT / f"snapugc_{SPLIT}_n{N}_seed{SEED}"
RUN.mkdir(parents=True, exist_ok=True)
manifest_path = RUN / "manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype={"video_id": str})
else:
    manifest = build_manifest(load_labels(LABELS), VIDEOS, n=N, seed=SEED)
    manifest.to_csv(manifest_path, index=False)
print(f"{len(manifest)} videos in this run")

t0 = time.time()
pilot = extractor.run(manifest.head(5), RUN / "work", table_name="network_timeseries_pilot")
per_video = (time.time() - t0) / max(pilot["video_id"].nunique(), 1)
print(f"⏱  ~{per_video:.0f} s per video → ~{per_video * len(manifest) / 3600:.1f} h for all {len(manifest)}")

deadline = NOTEBOOK_START + TIME_BUDGET_H * 3600
BATCH = 25
for start in range(0, len(manifest), BATCH):
    if time.time() + BATCH * per_video > deadline:
        print("⏸  Time budget reached; stopping early. Re-run (or commit again) to continue.")
        break
    extractor.run(manifest.iloc[start:start + BATCH], RUN / "work")

# One table over every finished video (cached predictions are reused, nothing is recomputed)
timeseries = extractor.run(manifest[[(RUN / "work" / "preds" / f"{v}.npz").exists() for v in manifest.video_id]],
                           RUN / "work")
print(f"✅ {timeseries['video_id'].nunique()}/{len(manifest)} videos processed")

## 6. Brain features vs baselines
For each target (NAWP = how much people watch, ECR = share who stay past 5 s), compares `baseline` (length + title/description stats), `brain` (TRIBE features) and `baseline+brain`, scored by Spearman correlation on held-out videos.

In [ ]:
from IPython.display import Image, Markdown, display
from neuroengage.pipeline import run_score_training

MIN_VIDEOS = 50   # below this the comparison is noise
done_ids = set(timeseries["video_id"])
if len(done_ids) < MIN_VIDEOS:
    print(f"Only {len(done_ids)} videos finished; need at least {MIN_VIDEOS} for a meaningful comparison. "
          "Run step 5 again (it resumes) before evaluating.")
else:
    res = run_score_training(
        manifest[manifest["video_id"].isin(done_ids)], timeseries, ["nawp", "ecr"], RUN / "results",
        baseline_cols=["duration_s", "title_n_words", "has_description"],
        hook_seconds=5, seed=SEED,
        title=f"SnapUGC ({len(done_ids)} videos): TRIBE v2 brain features vs baselines",
    )
    display(Image(str(RUN / "results" / "summary.png")))
    display(Markdown((RUN / "results" / "report.md").read_text()))


**Reading it:** the brain features help only if `baseline+brain` beats `baseline` by more than `spearman_std` in most folds (`folds_better_than_baseline` near 1.0). With a few hundred videos the error bars are wide, so scale `N` up before concluding anything.

## 7. Package results
Everything under `/kaggle/working` is saved with the notebook version (**Output** tab). This also writes a single zip for easy download.

In [ ]:
if (RUN / "results").exists():
    zip_path = shutil.make_archive(f"/kaggle/working/{RUN.name}_results", "zip", RUN / "results")
    print(zip_path, f"{Path(zip_path).stat().st_size / 1e6:.1f} MB")
else:
    print("No results yet (step 6 was skipped); predictions so far are saved under", RUN / "work")
print(shutil.disk_usage("/kaggle/working"))
